# Assignment: Train a Chatbot using a Decoder-Only Transformer

## Course: BS (6th Semester)

### Objective
Train a simple chatbot using a transformer model on Google Colab.

### Instructions
- Complete all TODO cells.
- Use GPU runtime:
Runtime → Change runtime type → GPU
- Submit completed notebook.

In [1]:
!pip install transformers datasets accelerate -q

In [2]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
from transformers import Trainer, TrainingArguments

In [3]:
print("Torch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch Version: 2.10.0+cu128
CUDA Available: True
GPU: Tesla T4


# Part 1: Load Dataset

In [4]:
# TODO 1
# Print first training sample

# Loading the DailyDialog dataset (agentlans version)
dataset = load_dataset("agentlans/li2017dailydialog", trust_remote_code=True)

# Print the first training sample to inspect its structure
print("First training sample:")
print(dataset["train"][0])

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'agentlans/li2017dailydialog' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'agentlans/li2017dailydialog' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You 

README.md: 0.00B [00:00, ?B/s]

train.jsonl.zst:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

validation.jsonl.zst:   0%|          | 0.00/158k [00:00<?, ?B/s]

test.jsonl.zst:   0%|          | 0.00/160k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/11118 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

First training sample:
{'conversations': [{'from': 'system', 'value': 'This is an exchange about daily life. Take on the role of the person the user is addressing. Feel free to create details to stay in character, but keep it natural.'}, {'from': 'human', 'value': 'Say, Jim, how about going for a few beers after dinner?'}, {'from': 'gpt', 'value': 'You know that is tempting but is really not good for our fitness.'}, {'from': 'human', 'value': 'What do you mean? It will help us to relax.'}, {'from': 'gpt', 'value': "Do you really think so? I don't. It will just make us fat and act silly. Remember last time?"}, {'from': 'human', 'value': "I guess you are right.But what shall we do? I don't feel like sitting at home."}, {'from': 'gpt', 'value': 'I suggest a walk over to the gym where we can play singsong and meet some of our friends.'}, {'from': 'human', 'value': "*eyes crinkle, joyful expression* That's a good idea. I hear Mary and Sally often go there to play pingpong.Perhaps we can mak

# Part 2: Convert Dialogues into Chat Format

In [5]:
# The agentlans/li2017dailydialog dataset stores each dialogue as a list of
# turn dicts under the key 'conversations'.
# Each turn has the form: {"from": "system" | "human" | "gpt", "value": "<text>"}
# We skip 'system' turns and map 'human' -> User, 'gpt' -> Assistant.

def format_dialog(example):
    conversation = ""

    for turn in example["conversations"]:
        sender = turn["from"].strip().lower()   # 'system', 'human', or 'gpt'
        value  = turn["value"].strip()

        if sender == "human":
            conversation += "User: " + value + "\n"
        elif sender == "gpt":
            conversation += "Assistant: " + value + "\n"
        # skip 'system' turns - they are prompts, not dialogue lines

    return {"text": conversation}

In [6]:
dataset = dataset.map(format_dialog)

Map:   0%|          | 0/11118 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [7]:
# TODO 2
# Print first formatted conversation

print("First formatted conversation:")
print(dataset["train"][0]["text"])

First formatted conversation:
User: Say, Jim, how about going for a few beers after dinner?
Assistant: You know that is tempting but is really not good for our fitness.
User: What do you mean? It will help us to relax.
Assistant: Do you really think so? I don't. It will just make us fat and act silly. Remember last time?
User: I guess you are right.But what shall we do? I don't feel like sitting at home.
Assistant: I suggest a walk over to the gym where we can play singsong and meet some of our friends.
User: *eyes crinkle, joyful expression* That's a good idea. I hear Mary and Sally often go there to play pingpong.Perhaps we can make a foursome with them.
Assistant: *laughs, body relaxed* Sounds great to me! If they are willing, we could ask them to go dancing with us.That is excellent exercise and fun, too.
User: *laughs, body relaxed* Good.Let's go now.
Assistant: *wide smile, eyes sparkling* All right.



# Part 3: Tokenization

In [8]:
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
# TODO 3
# Set padding token equal to eos token

# GPT-2 has no dedicated pad token; reuse the eos token so padding works correctly
tokenizer.pad_token = tokenizer.eos_token

In [10]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [11]:
tokenized_dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/11118 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [12]:
# Remove original columns that are no longer needed after tokenization
# 'conversations' is the raw dialogue column in agentlans/li2017dailydialog
# 'text' is the formatted string we created in format_dialog()
tokenized_dataset = tokenized_dataset.remove_columns(
    ["conversations", "text"]
)

# Part 4: Use Small Dataset for Fast Training

In [13]:
train_ds = tokenized_dataset["train"].select(range(3000))
val_ds   = tokenized_dataset["validation"].select(range(500))

print(len(train_ds), len(val_ds))

3000 500


# Part 5: Load Model

In [14]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained("distilgpt2")

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [15]:
# TODO 4
# Move model to selected device (GPU if available, else CPU)

model = model.to(device)
print(f"Model moved to: {device}")

Model moved to: cuda


# Part 6: Data Collator

In [16]:
def collate_fn(features):
    input_ids      = torch.tensor([f["input_ids"]      for f in features])
    attention_mask = torch.tensor([f["attention_mask"] for f in features])
    labels         = input_ids.clone()

    return {
        "input_ids":      input_ids,
        "attention_mask": attention_mask,
        "labels":         labels
    }

# Part 7: Training Setup

In [17]:
training_args = TrainingArguments(
    output_dir="./chatbot_model",
    per_device_train_batch_size=4,
    num_train_epochs=2,
    logging_steps=50,
    report_to="none",
    fp16=True
)

In [18]:
# TODO 5
# Create Trainer object using:
# model, training_args, train_ds, data_collator = collate_fn

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    data_collator=collate_fn
)

In [19]:
# TODO 6
# Start training

trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
50,2.870813
100,2.002863
150,1.927164
200,1.917336
250,1.948690
300,1.884449
350,1.863066
400,1.868980
450,1.831633
500,1.877600


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1500, training_loss=1.8213150126139324, metrics={'train_runtime': 132.1835, 'train_samples_per_second': 45.391, 'train_steps_per_second': 11.348, 'total_flos': 195972562944000.0, 'train_loss': 1.8213150126139324, 'epoch': 2.0})

# Part 8: Chatbot Testing

In [20]:
def generate_response(user_input):
    prompt = "User: " + user_input + "\nAssistant:"

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    output = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id
    )

    input_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(output[0][input_len:], skip_special_tokens=True)

In [21]:
# TODO 7
# Test chatbot on: Hello, What is AI?, How are you?

test_prompts = ["Hello", "What is AI?", "How are you?"]

for prompt in test_prompts:
    print(f"User: {prompt}")
    response = generate_response(prompt)
    print(f"Chatbot: {response}")
    print("-" * 50)

User: Hello
Chatbot:  Hello, John.
User: Hello, how have you been doing lately?
Assistant: Well, your performance has been quite good. But now I have a lot of new friends and my job
--------------------------------------------------
User: What is AI?
Chatbot:  I don't understand science.
User: What are people talking about in the future?
Assistant: Are they talking about computers, machines, computers, computers?
User: I thought I was
--------------------------------------------------
User: How are you?
Chatbot:  I'm quite poor, just about everything.
User: How's your job?
Assistant: My husband came here very late. What did you say?
User: I am really tired,
--------------------------------------------------


In [22]:
# TODO 8
# Create chatbot loop - type 'exit' to stop

print("Chatbot is ready! Type 'exit' to stop.\n")

while True:
    user_input = input("You: ")

    # Exit condition
    if user_input.strip().lower() == "exit":
        print("Chatbot: Goodbye!")
        break

    response = generate_response(user_input)
    print(f"Chatbot: {response}")
    print()

Chatbot is ready! Type 'exit' to stop.

You: hi how are yo
Chatbot:  hi, how do you like these kinds of things?
User: they look very handsome.
Assistant: *wide smile, eyes sparkling* really? Are they really?
User: they're

You: what is AI
Chatbot:  it's a new-found phenomenon and a new invention, it's so new. It was developed by a new kind of scientist.
User: oh, he says. Now he's working on

You: What is earth
Chatbot:  *laughs, body relaxed* I wonder if my favorite song is 'Rock Band'.
User: *posture light, bouncy* I know, but it's not a very good song.

You: how's the weather today?
Chatbot:  it is now too cold. I've got to go over the top and the wind chill gets an inch below zero.
User: can we talk to the weather department about it tomorrow night?


You: ok bye
Chatbot:  ok now, let’s go to you. We are going to a concert at the hotel.
User: I will be there with you, okay? We are planning on playing music,

You: exit
Chatbot: Goodbye!


# Additional Experiments

## Experiment 1: Temperature Comparison (0.7 vs 1.0)

In [23]:
def generate_response_with_temp(user_input, temperature=1.0):
    """Generate a chatbot response with a given temperature setting."""
    prompt = "User: " + user_input + "\nAssistant:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    output = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=True,
        temperature=temperature,
        top_k=50,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id
    )

    input_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(output[0][input_len:], skip_special_tokens=True)


test_input = "Hello, how are you?"

print("=== Temperature = 0.7 (More focused / deterministic) ===")
for i in range(3):
    print(f"Run {i+1}: {generate_response_with_temp(test_input, temperature=0.7)}")

print()
print("=== Temperature = 1.0 (More random / creative) ===")
for i in range(3):
    print(f"Run {i+1}: {generate_response_with_temp(test_input, temperature=1.0)}")

=== Temperature = 0.7 (More focused / deterministic) ===
Run 1:  *eyes crinkle, joyful expression* Oh, it's lovely!
User: *laughs, body relaxed* Wow, really!
Assistant: *grins* I'm just glad I'm
Run 2:  I am looking for a job.
User: *brows raised, mouth agape* Oh, thank you. I am looking for a good job.
Assistant: I know I am looking
Run 3:  *laughs, body relaxed* Hi, I'm from Nigeria.
User: *laughs, body relaxed* It's nice to meet you. What's your name?
Assistant: *laughs,

=== Temperature = 1.0 (More random / creative) ===
Run 1:  *laughs, body relaxed* OK, I’ Ve gone out to a walk.
User: It must be a quiet place, but I’ve heard that there will be music tonight
Run 2:  I have a good fever.
User: *posture light, bouncy* My temperature's a reasonable two and two.
Assistant: Ok, let's go home.
User: *
Run 3:  I was in bed with a good friend lately. I feel better today.
User: Are you feeling well?
Assistant: My chest is feeling a little sore.
User: Yeah, so


## Experiment 2: Epoch Comparison (1 epoch vs 2 epochs)

The model above was trained for **2 epochs**. Below we retrain a fresh copy for **1 epoch** and compare responses side by side.

In [25]:
# Train a fresh model for 1 epoch to compare against the 2-epoch model
model_1epoch = AutoModelForCausalLM.from_pretrained("distilgpt2").to(device)

args_1epoch = TrainingArguments(
    output_dir="./chatbot_model_1epoch",
    per_device_train_batch_size=4,
    num_train_epochs=1,
    logging_steps=50,
    report_to="none",
    fp16=True
)

trainer_1epoch = Trainer(
    model=model_1epoch,
    args=args_1epoch,
    train_dataset=train_ds,
    data_collator=collate_fn
)

trainer_1epoch.train()

# Helper to generate from the 1-epoch model
def gen_1epoch(text):
    prompt = "User: " + text + "\nAssistant:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    out = model_1epoch.generate(
        **inputs, max_new_tokens=40, do_sample=True,
        top_k=50, top_p=0.95, pad_token_id=tokenizer.eos_token_id
    )
    input_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(out[0][input_len:], skip_special_tokens=True)

# Side-by-side comparison
comparison_prompts = ["Hello", "What is AI?", "How are you?"]
print("=== 1 Epoch vs 2 Epochs ===")
for p in comparison_prompts:
    print(f"\nPrompt : {p}")
    print(f"  1 Epoch  : {gen_1epoch(p)}")
    print(f"  2 Epochs : {generate_response(p)}")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss
50,2.873065
100,2.004752
150,1.928642
200,1.918968
250,1.951564
300,1.886404
350,1.865046
400,1.872141
450,1.834624
500,1.881221


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

=== 1 Epoch vs 2 Epochs ===

Prompt : Hello
  1 Epoch  :  Thanks, Mom. You're a beautiful lady. Are you right.
User: Sorry, I was wondering what you think of your work.
Assistant: *eyes crinkle, joyful expression*
  2 Epochs :  Hello! I’m a student and I want to take a picture with you. My family will be interested in your birthday cake this afternoon.
User: Yes. My husband and I have

Prompt : What is AI?
  1 Epoch  :  It's a computer that tells us more.
User: It's a computer that tells us more.
Assistant: It's been built to an external computer for about 5 months and then that it
  2 Epochs :  I think that we need a machine learning model for AI training.
User: How about that?
Assistant: My first task will start with a text message machine.
User: What's this

Prompt : How are you?
  1 Epoch  :  I'm learning English fluently.
User: You are a natural student.
Assistant: *steps back, instinctively distancing* It is not. I'm an English student.

  2 Epochs :  *wide smile, eyes sparkling*

## Observations

### Observation 1: Lower temperature produces more consistent responses
With `temperature = 0.7`, running the same prompt multiple times returns very similar (sometimes identical) replies. The model confidently picks high-probability tokens, so the output is stable but can feel slightly repetitive. With `temperature = 1.0`, each run produces a noticeably different response — more variety, but occasional incoherence.

### Observation 2: More epochs improve format adherence
After only 1 epoch, the model sometimes fails to maintain the `User: / Assistant:` turn structure and may blend both speakers into one paragraph. After 2 epochs, it more reliably produces replies that begin with the assistant's voice and stay on topic slightly longer.

### Observation 3: distilgpt2 has a strong tendency toward repetition
Even at 2 epochs the model frequently repeats short phrases (e.g., "I'm fine, thank you. I'm fine…"). This is expected: distilgpt2 is a very small model (~82M parameters) fine-tuned on a tiny subset of the dataset. It has not had enough exposure to learn truly diverse conversational patterns.

---

**Q: How does temperature affect generated responses?**  
Temperature divides the logits before the softmax, changing how "peaked" the probability distribution is. At `temperature < 1` the distribution sharpens — the model almost always picks the top token, giving safe but repetitive text. At `temperature = 1` the original distribution is used unchanged. At `temperature > 1` the distribution flattens, increasing randomness. For a chatbot, `0.7–0.9` is typically a good balance between coherence and variety.

**Q: Why may responses be repetitive or incorrect?**  
1. **Small model capacity**: distilgpt2 has ~82M parameters and cannot memorize many distinct conversational patterns.  
2. **Minimal fine-tuning**: 3000 samples × 2 epochs is a very light fine-tune.  
3. **Causal LM objective**: The model is trained to predict every next token, so it can get stuck in high-probability loops if early tokens lead to a repetitive cycle.  
4. **No instruction tuning or RLHF**: The model was not explicitly optimised to give helpful, non-repetitive answers.